In [ ]:
#For loading and manipulating NIfTI medical image files
import nibabel as nib
#For efficient numerical operations and array handling
import numpy as np
#For interacting with the file system
import os
#For plotting histograms and visualisations
import matplotlib.pyplot as plt

In [ ]:
#Load a NIfTI image from the specified path
#Returns both the nibabel object (for metadata access) and the image data array
def load_nifti(file_path):
    nii = nib.load(file_path)
    data = nii.get_fdata()
    return nii, data

In [ ]:
#Apply Z-score normalisation to standardise the data to zero mean and unit variance
#Useful when comparing across scans with different intensity ranges
def zscore_normalisation(data):
    mean = np.mean(data)
    std = np.std(data)
    normalised_data = (data - mean) / std
    return normalised_data

In [ ]:
#Rescale data to the range [0, 1] using min-max normalisation
#Often used when consistent scaling is required for intensity-based models
def minmax_normalisation(data):
    min_val = np.min(data)
    max_val = np.max(data)
    normalised_data = (data - min_val) / (max_val - min_val)
    return normalised_data

In [ ]:
if __name__ == "__main__":

    #Dictionary to collect intensity values by shape for histogram plotting later
    shape_histograms = {}
    
    #Define input directories for different preprocessed scan types
    input_folders = ["mri_ad_BET", "mri_cn_BET", "pet_ad_BET", "pet_cn_BET"]

    #Map each input folder to a corresponding output folder for saving normalised images
    output_folder_map = {
        "mri_ad_BET": "resampled_mri_ad_intenorm",
        "mri_cn_BET": "resampled_mri_cn_intenorm",
        "pet_ad_BET": "resampled_pet_ad_intenorm",
        "pet_cn_BET": "resampled_pet_cn_intenorm"}

    for in_folder in input_folders:
        print(f"Processing folder: {in_folder}")
        out_folder = output_folder_map[in_folder]
        #Create the output directory if it does not already exist
        os.makedirs(out_folder, exist_ok=True)

        for filename in os.listdir(in_folder):
            if filename.endswith(".nii") or filename.endswith(".nii.gz"):
                file_path = os.path.join(in_folder, filename)
                #Load the NIfTI image and extract its data
                nii, data = load_nifti(file_path)
                print(f"Processing file: {file_path}")
                print(f"Image shape: {data.shape}")
            
                #Apply Z-score normalisation to standardise intensity distribution
                normalised_data = zscore_normalisation(data)
                
                #Save the normalised image to the appropriate output folder
                output_filename = "normalised_" + filename
                output_path = os.path.join(out_folder, output_filename)
                normalised_nii = nib.Nifti1Image(normalised_data, affine=nii.affine)
                nib.save(normalised_nii, output_path)
                print(f"Saved: {output_path}")

                #Store intensity values by shape for visual summary
                shape_key = str(data.shape)
                if shape_key not in shape_histograms:
                    shape_histograms[shape_key] = []
                shape_histograms[shape_key].append(normalised_data.flatten())
                
    #Create a combined histogram showing intensity distributions grouped by image shape
    plt.figure(figsize=(10, 6))
    for shape_key, data_list in shape_histograms.items():
        combined_data = np.concatenate(data_list)
        plt.hist(combined_data, bins=100, alpha=0.5, label=f"Shape {shape_key}")

    plt.title("Combined Histogram by Shape")
    plt.xlabel("Intensity")
    plt.ylabel("Count")
    plt.legend()
    plt.show()